# Setup


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -qU transformers datasets optimum
!pip install -qU openai wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 654, done.
remote: Counting objects: 100% (654/654), done.
remote: Compressing objects: 100% (492/492), done.
remote: Total 654 (delta 156), reused 420 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (654/654), 5.29 MiB | 7.25 MiB/s, done.
Resolving deltas: 100% (156/156), done.
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Imports


In [2]:
from google.colab import userdata
import wandb
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests
import pandas as pd
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
device = "cuda"
torch_dtype = None
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
wandb.login(key=userdata.get('wandb'))
login(token=userdata.get('HF_TOKEN'))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sabryabdallah37 (sabryabdallah37-helwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
df_training = pd.read_json('/content/drive/MyDrive/opinion-mining/train_labeled_v2.json', lines=True)
df_training.head()

,input,prediction
0,I charge it at night and skip taking the cord ...,"{""domain"":""electronics"",""aspects"":[{""term"":""Ba..."
1,I bought a HP Pavilion DV41222nr laptop and ha...,"{""domain"":""electronics"",""aspects"":[{""term"":""HP..."
2,The tech guy then said the service center does...,"{""domain"":""electronics"",""aspects"":[{""term"":""se..."
3,I investigated netbooks and saw the Toshiba NB...,"{""domain"":""electronics"",""aspects"":[{""term"":""Ne..."
4,The other day I had a presentation to do for a...,"{""domain"":""software"",""aspects"":[{""term"":""compu..."


In [15]:
df_testing = pd.read_json('/content/drive/MyDrive/opinion-mining/test_labeled_v2.json')
df_testing.head()

,input,prediction
0,"Boot time is super fast, around anywhere from ...","{""domain"":""electronics"",""aspects"":[{""term"":""Bo..."
1,tech support would not fix the problem unless ...,"{""domain"":""software"",""aspects"":[{""term"":""tech ..."
2,but in resume this computer rocks!,"{""domain"":""electronics"",""aspects"":[{""term"":""co..."
3,Set up was easy.,"{""domain"":""general"",""aspects"":[{""term"":""Set up..."
4,Did not enjoy the new Windows 8 and touchscree...,"{""domain"":""software"",""aspects"":[{""term"":""Windo..."


# Formatting Finetune Data

In [ ]:
system_message = "\n".join(["You extract structured information from text and return only valid JSON.",
                            "",
                            "No explanation No introduction No conclusion"])

In [ ]:
llm_finetunning_data = []

for i, row in df_training.iterrows():
    input = row['input']
    output = row['prediction']

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "\n".join(["Extract the domain and all aspect terms with their sentiment polarity.",
                                  "- Domains: electronics, restaurants, movies, books, software, general",
                                  "- Polarity: positive, negative, neutral",
                                  "- Extract ALL aspects mentioned in the text",
                                  "input :",
                                  input,
                                  "output format:",
                                  '{"domain":"...","aspects":[{"term":"...","polarity":"..."}, ...]}']),
        "input" : "",
        "output" : output,
        "history" : []
    })

random.Random(101).shuffle(llm_finetunning_data)


In [ ]:
len(llm_finetunning_data)

5959

In [ ]:
data_dir = "/content/drive/MyDrive/opinion-mining"

train_sample_sz = 5000

train_ds = llm_finetunning_data[:train_sample_sz]
eval_ds = llm_finetunning_data[train_sample_sz:]

os.makedirs(join(data_dir, "datasets", "llamafactory-finetune-data"), exist_ok=True)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [ ]:
join(data_dir, "datasets", "llamafactory-finetune-data", "val.json")

'/content/drive/MyDrive/opinion-mining/datasets/llamafactory-finetune-data/val.json'

# Llama-Factory Configuration

In [ ]:
# # Configure LLaMA-Factory for the new datasets

# # update /content/LLaMA-Factory/data/dataset_info.json and append
# ```
  "opinion_mining_finetune_train": {
        "file_name": "/content/drive/MyDrive/opinion-mining/datasets/llamafactory-finetune-data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "opinion_mining_finetune_val": {
        "file_name": "/content/drive/MyDrive/opinion-mining/datasets/llamafactory-finetune-data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
# ```

In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: opinion_mining_finetune_train
eval_dataset: opinion_mining_finetune_val
template: qwen
cutoff_len: 1000
##max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
resume_from_checkpoint: /content/drive/MyDrive/opinion-mining/models/checkpoint-1000
output_dir: /content/drive/MyDrive/opinion-mining/models/
logging_steps: 25
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 550
#disable_tqdm: true

report_to: wandb
run_name: opinion-mining-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "AbdallahSabry1/opinion-mining-model-qwen2.5"
hub_private_repo: true
hub_strategy: checkpoint

Writing /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


# Finetuning

In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-04-24 17:32:12] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:670] 2026-04-24 17:32:13,136 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:742] 2026-04-24 17:32:13,138 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_i

# New Finetuned Model responses

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
finetuned_model_id = "/content/drive/MyDrive/opinion-mining/models/checkpoint-3750"
model.load_adapter(finetuned_model_id)
model.to(device)

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=1536, out_features=32, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=32, out_features=1536, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=1536, out_features=256, bias=True)
            (lora_dropout): ModuleDict(
              (default): Identity()
         

In [5]:
def run_qwen(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=65,
            do_sample=False,
            temperature = 0.0,
            top_k = None,
            top_p = None
        )

    result = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return result.strip()

In [6]:
prompt = "\n".join(["You extract structured information from text and return only valid JSON.",
                            "",
                            "No explanation No introduction No conclusion",
                            "Extract the domain and all aspect terms with their sentiment polarity.",
                                  "- Domains: electronics, restaurants, movies, books, software, general",
                                  "- Polarity: positive, negative, neutral",
                                  "- Extract ALL aspects mentioned in the text",
                                  "input :",
                                  "This laptop features a high solution display and long battery life. Loved the comfortable keyboard and touch pad, also very light weight so it’s easy to carry around.",
                                  "output format:",
                                  '{"domain":"...","aspects":[{"term":"...","polarity":"..."}, ...]}'
                        ])

In [9]:
print(run_qwen(prompt))

{"domain":"electronics","aspects":[{"term":"Solution display","polarity":"positive"},{"term":"Battery life","polarity":"positive"},{"term":"Comfortable keyboard","polarity":"positive"},{"term":"Touch pad","polarity":"positive"},{"term":"Weight","polarity":"positive"}]}


In [16]:
results = []

for i, row in tqdm(df_testing.iterrows(), total=len(df_testing), desc="Running Qwen inference"):
    text = row['input']

    prompt = "\n".join(["You extract structured information from text and return only valid JSON.",
                            "",
                            "No explanation No introduction No conclusion",
                            "Extract the domain and all aspect terms with their sentiment polarity.",
                                  "- Domains: electronics, restaurants, movies, books, software, general",
                                  "- Polarity: positive, negative, neutral",
                                  "- Extract ALL aspects mentioned in the text",
                                  "input :",
                                  text,
                                  "output format:",
                                  '{"domain":"...","aspects":[{"term":"...","polarity":"..."}, ...]}'
                        ])

    prediction = run_qwen(prompt)

    results.append({
        "input": text,
        "prediction": prediction
    })

Running Qwen inference:   0%|          | 0/1572 [00:00<?, ?it/s]

In [17]:
import json

with open("finetuned_LLM_responses.json", "w") as f:
    json.dump(results, f, indent=2)

In [18]:
import shutil

shutil.copy(
    "finetuned_LLM_responses.json",
    "/content/drive/MyDrive/opinion-mining/finetuned_LLM_responses.json"
)

'/content/drive/MyDrive/opinion-mining/finetuned_LLM_responses.json'